In [ ]:
print("Hello world!")

In [10]:
# -*- coding: utf-8 -*-
"""
Генерация данных для лабораторной работы №5 (выборочное наблюдение).
10 вариантов, по 500 счетов-фактур.
Создаёт:
  - lab5_invoices.xlsx  (10 листов: Вариант 1 ... Вариант 10)
  - lab5_answers.docx   (ожидаемые результаты по каждому варианту)
"""

import random
import numpy as np
import pandas as pd
from docx import Document
from docx.shared import Pt
from datetime import datetime, timedelta

# ---------------------------------------------------------------------------
# 1. ОПИСАНИЕ ВАРИАНТОВ
# ---------------------------------------------------------------------------
# Для каждого варианта: средняя, дисперсия (σ²), доля нарушений, доля неоплаченных,
# ключевая особенность, список контрагентов (первые 8 используются для аномалий,
# остальные — как общий пул названий).

VARIANTS = {
    1: {
        "mean": 57000, "var": 2.23e8, "p_viol": 0.20, "p_unpaid": 0.15,
        "feature": "Базовый",
        "contractors": [
            "ООО «Ромашка»", "АО «Вектор»", "ИП Иванов И.И.", "ООО «Север»",
            "ООО «Заря»", "АО «Прогресс»", "ООО «Кристалл»", "ИП Петров П.П.",
            "ООО «Альфа»", "ООО «Бета»", "АО «Гамма»", "ООО «Дельта»",
        ],
    },
    2: {
        "mean": 50000, "var": 8.2e7, "p_viol": 0.10, "p_unpaid": 0.10,
        "feature": "Низкий разброс",
        "contractors": [
            "ООО «СтройМонтаж»", "АО «ГлавСтрой»", "ООО «БетонСтрой»",
            "ООО «Кирпич-Торг»", "ИП Строев С.С.", "ООО «Панель-Сервис»",
            "ООО «Фундамент»", "АО «Высота»", "ООО «Кран-Сервис»",
            "ООО «Леса-Торг»", "ИП Плотников П.П.", "ООО «СтройБаза»",
        ],
    },
    3: {
        "mean": 60000, "var": 5.34e8, "p_viol": 0.30, "p_unpaid": 0.25,
        "feature": "Высокий разброс",
        "contractors": [
            "ООО «ТоргСервис»", "АО «РитейлГрупп»", "ООО «ОптТорг»",
            "ООО «Розница-Плюс»", "ИП Торговцев Т.Т.", "ООО «ПродуктТорг»",
            "ООО «МясоТорг»", "АО «РыбТорг»", "ООО «ОвощТорг»",
            "ООО «ФруктТорг»", "ИП Ягодкин Я.Я.", "ООО «МолокоТорг»",
        ],
    },
    4: {
        "mean": 60000, "var": 3e8, "p_viol": 0.40, "p_unpaid": 0.30,
        "feature": "Много нарушений",
        "contractors": [
            "ООО «Логистик-1»", "АО «ТрансЛогистик»", "ООО «СкладСервис»",
            "ООО «Грузоперевозки»", "ИП Перевозчиков П.П.", "ООО «Экспресс-Доставка»",
            "ООО «Курьер-Сервис»", "АО «Почта-Логистик»", "ООО «Склад-Торг»",
            "ООО «Терминал-Плюс»", "ИП Логистов Л.Л.", "ООО «Карго-Сервис»",
        ],
    },
    5: {
        "mean": 55000, "var": 1.42e8, "p_viol": 0.05, "p_unpaid": 0.05,
        "feature": "Мало нарушений",
        "contractors": [
            "ООО «АйТи-Сервис»", "АО «Софт-Групп»", "ООО «Хард-Сервис»",
            "ООО «Веб-Студия»", "ИП Программистов П.П.", "ООО «Девелопер-Сервис»",
            "ООО «Аналитик-Софт»", "АО «Дата-Центр»", "ООО «Клауд-Сервис»",
            "ООО «Секьюрити-Софт»", "ИП Разработчиков Р.Р.", "ООО «Тест-Софт»",
        ],
    },
    6: {
        "mean": 80000, "var": 1.71e9, "p_viol": 0.20, "p_unpaid": 0.15,
        "feature": "Много крупных",
        "contractors": [
            "ООО «МедСервис»", "АО «ФармГрупп»", "ООО «Аптека-Сервис»",
            "ООО «МедТех-Торг»", "ИП Врачов В.В.", "ООО «Клиника-Плюс»",
            "ООО «Диагностика-Сервис»", "АО «Лаборатория-Групп»",
            "ООО «Анализ-Сервис»", "ООО «МедОборудование»", "ИП Медсестров М.М.",
            "ООО «Стоматология-Сервис»",
        ],
    },
    7: {
        "mean": 30000, "var": 3.45e7, "p_viol": 0.15, "p_unpaid": 0.10,
        "feature": "Много мелких",
        "contractors": [
            "ООО «Завод-1»", "АО «Металлург»", "ООО «Прокат-Сервис»",
            "ООО «Литьё-Сервис»", "ИП Металлистов М.М.", "ООО «Прокат-Плюс»",
            "ООО «Литьё-Плюс»", "АО «Сталь-Групп»", "ООО «Чугун-Сервис»",
            "ООО «Алюминий-Сервис»", "ИП Алюминиев А.А.", "ООО «Медь-Сервис»",
        ],
    },
    8: {
        "mean": 55000, "var": 2e8, "p_viol": 0.20, "p_unpaid": 0.15,
        "feature": "Сильная связь (90% неоплаченных с нарушениями)",
        "contractors": [
            "ООО «ФинСервис»", "АО «Банк-Групп»", "ООО «Кредит-Сервис»",
            "ООО «Депозит-Сервис»", "ИП Банкиров Б.Б.", "ООО «Инвест-Плюс»",
            "ООО «Фонд-Сервис»", "АО «Страх-Групп»", "ООО «Лизинг-Сервис»",
            "ООО «Факторинг-Сервис»", "ИП Факторингов Ф.Ф.", "ООО «Брокер-Сервис»",
        ],
    },
    9: {
        "mean": 55000, "var": 2.35e8, "p_viol": 0.20, "p_unpaid": 0.15,
        "feature": "Нет связи (нарушения равномерны)",
        "contractors": [
            "ООО «Транс-1»", "АО «АвтоТранс»", "ООО «ЖД-Сервис»",
            "ООО «Авиа-Сервис»", "ИП Перевозчиков П.П.", "ООО «Мор-Сервис»",
            "ООО «Реч-Сервис»", "АО «Труб-Групп»", "ООО «Газ-Сервис»",
            "ООО «Нефть-Сервис»", "ИП Нефтяников Н.Н.", "ООО «Хим-Сервис»",
        ],
    },
    10: {
        "mean": 55000, "var": 1.79e8, "p_viol": 0.20, "p_unpaid": 0.15,
        "feature": "Смещённая механическая выборка (упорядочено по сумме)",
        "contractors": [
            "ООО «Агро-1»", "АО «АгроХолдинг»", "ООО «Зерно-Сервис»",
            "ООО «Мясо-Сервис»", "ИП Фермеров Ф.Ф.", "ООО «Молоко-Сервис»",
            "ООО «Птица-Сервис»", "АО «Свиновод-Групп»", "ООО «Коров-Сервис»",
            "ООО «Овца-Сервис»", "ИП Овцеводов О.О.", "ООО «Коза-Сервис»",
        ],
    },
}

# ---------------------------------------------------------------------------
# 2. ГЕНЕРАЦИЯ 500 СЧЕТОВ ДЛЯ ВАРИАНТА
# ---------------------------------------------------------------------------
def generate_invoices(variant_no: int, seed: int = None) -> pd.DataFrame:
    cfg = VARIANTS[variant_no]
    rng = np.random.default_rng(seed if seed is not None else 1000 + variant_no)
    random.seed(seed if seed is not None else 1000 + variant_no)

    N = 500
    mean = cfg["mean"]
    var = cfg["var"]
    print(f'величина дисперсии при генерации данных варианта {variant_no} = {var}')
    sigma = np.sqrt(var)
    p_viol = cfg["p_viol"]
    p_unpaid = cfg["p_unpaid"]
    contractors = cfg["contractors"]

    # --- суммы: логнормальное распределение, подогнанное под mean и var ---
    # Для логнормального: mean = exp(mu + sigma_ln^2/2), var = (exp(sigma_ln^2)-1)*exp(2mu+sigma_ln^2)
    # Решаем относительно mu и sigma_ln.
    sigma_ln2 = np.log(1 + var / mean**2)
    sigma_ln = np.sqrt(sigma_ln2)
    mu_ln = np.log(mean) - sigma_ln2 / 2
    sums = rng.lognormal(mean=mu_ln, sigma=sigma_ln, size=N)

    # округляем до сотен
    sums = np.round(sums / 100) * 100
    sums = np.clip(sums, 5000, 500000)

    # --- аномалии ---

    # Аномалия 2: 3 счёта с нулевой суммой
    zero_idx = rng.choice([i for i in range(N)], size=3, replace=False)
    sums[zero_idx] = 0

    # --- статус оплаты ---
    # p_unpaid — доля неоплаченных; частично — столько же; остальное — оплачено
    n_unpaid = int(round(p_unpaid * N))
    n_partial = n_unpaid
    n_paid = N - n_unpaid - n_partial

    statuses = ["Оплачен"] * n_paid + ["Не оплачен"] * n_unpaid + ["Частично"] * n_partial
    rng.shuffle(statuses)
    statuses = np.array(statuses)

    # --- нарушения ---
    violations = np.array(["Нет"] * N, dtype=object)
    n_viol = int(round(p_viol * N))

    if variant_no == 8:
        # Сильная связь: 90% неоплаченных имеют нарушения
        unpaid_idx = np.where(statuses == "Не оплачен")[0]
        n_unpaid_viol = int(round(0.9 * len(unpaid_idx)))
        viol_unpaid = rng.choice(unpaid_idx, size=n_unpaid_viol, replace=False)
        violations[viol_unpaid] = "Да"
        # остальные нарушения — среди оплаченных/частично
        remaining = n_viol - n_unpaid_viol
        pool = np.setdiff1d(np.arange(N), viol_unpaid)
        viol_other = rng.choice(pool, size=max(remaining, 0), replace=False)
        violations[viol_other] = "Да"
    elif variant_no == 9:
        # Нет связи: нарушения равномерно по статусам
        for st in ["Оплачен", "Не оплачен", "Частично"]:
            idx = np.where(statuses == st)[0]
            n_st_viol = int(round(p_viol * len(idx)))
            if n_st_viol > 0:
                viol_st = rng.choice(idx, size=n_st_viol, replace=False)
                violations[viol_st] = "Да"
    else:
        # Базовая связь: 73% неоплаченных с нарушениями (как в описании)
        unpaid_idx = np.where(statuses == "Не оплачен")[0]
        n_unpaid_viol = int(round(0.73 * len(unpaid_idx)))
        viol_unpaid = rng.choice(unpaid_idx, size=n_unpaid_viol, replace=False) if n_unpaid_viol > 0 else np.array([], dtype=int)
        violations[viol_unpaid] = "Да"
        remaining = n_viol - len(viol_unpaid)
        pool = np.setdiff1d(np.arange(N), viol_unpaid)
        if remaining > 0:
            viol_other = rng.choice(pool, size=min(remaining, len(pool)), replace=False)
            violations[viol_other] = "Да"

    # --- контрагенты ---
    # повторяем список, чтобы хватило на 500 строк
    contractors_full = (contractors * (N // len(contractors) + 1))[:N]
    # для аномалий подставим первых 8 контрагентов
    for k, idx in enumerate(list(zero_idx)):
        contractors_full[idx] = contractors[k % len(contractors)]

    # --- даты ---
    start = datetime(2024, 1, 1)
    dates = [start + timedelta(days=int(rng.integers(0, 365))) for _ in range(N)]

    # --- номера счетов ---
    numbers = list(range(1, N + 1))

    df = pd.DataFrame({
        "Номер счёта": numbers,
        "Дата": [d.strftime("%d.%m.%Y") for d in dates],
        "Контрагент": contractors_full,
        "Сумма счёта": sums.astype(int),
        "Статус оплаты": statuses,
        "Наличие нарушений": violations,
    })

    # Для варианта 10 — упорядочить по сумме (для механической выборки)
    if variant_no == 10:
        df = df.sort_values("Сумма счёта").reset_index(drop=True)
        df["Номер счёта"] = range(1, N + 1)

    return df


# ---------------------------------------------------------------------------
# 3. РАСЧЁТ ПОКАЗАТЕЛЕЙ
# ---------------------------------------------------------------------------
def compute_stats(df: pd.DataFrame):
    """Возвращает словарь с показателями генеральной совокупности."""
    sums = df["Сумма счёта"].values
    return {
        "mean": float(np.mean(sums)),
        "var": float(np.var(sums, ddof=0)),
        "std": float(np.std(sums, ddof=0)),
        "min": int(np.min(sums)),
        "max": int(np.max(sums)),
    }


def mechanical_sample(df: pd.DataFrame, n: int = 50):
    """Механическая выборка: каждый k-й элемент."""
    N = len(df)
    k = N // n
    idx = np.arange(0, N, k)[:n]
    return df.iloc[idx].copy()


def random_sample(df: pd.DataFrame, n: int = 50, seed: int = 42):
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(df), size=n, replace=False)
    return df.iloc[idx].copy()


def sample_stats(sample: pd.DataFrame):
    sums = sample["Сумма счёта"].values
    n = len(sums)
    mean = float(np.mean(sums))
    var = float(np.var(sums, ddof=1))  # выборочная дисперсия
    std = float(np.sqrt(var))
    mu = std / np.sqrt(n)
    delta = 2 * mu  # t=2 для доверительной вероятности ~0.954
    return {
        "n": n,
        "mean": mean,
        "var": var,
        "std": std,
        "mu": mu,
        "delta": delta,
        "lower": mean - delta,
        "upper": mean + delta,
    }


def required_sample_size(var: float, delta: float, t: float = 2.0):
    print(f'Дисперсия для объема выборки = {var}')
    return int(np.ceil((t**2 * var * 500) / (delta**2 * 500 + var * t ** 2)))


def risk_level(sample: pd.DataFrame):
    """Уровень риска по сумме счёта."""
    sums = sample["Сумма счёта"].values
    high = np.sum(sums > 100000)
    medium = np.sum((sums >= 50000) & (sums <= 100000))
    low = np.sum(sums < 50000)
    total = len(sums)
    return {
        "high": int(high), "medium": int(medium), "low": int(low),
        "high_pct": round(100 * high / total, 1),
        "medium_pct": round(100 * medium / total, 1),
        "low_pct": round(100 * low / total, 1),
    }


def pivot_status_violations(df: pd.DataFrame):
    """Сводная таблица: статус оплаты × нарушения."""
    ct = pd.crosstab(df["Статус оплаты"], df["Наличие нарушений"])
    ct = ct.reindex(index=["Оплачен", "Не оплачен", "Частично"],
                    columns=["Да", "Нет"], fill_value=0)
    ct["Всего"] = ct.sum(axis=1)
    total = ct.sum(axis=0)
    total.name = "Итого"
    ct = pd.concat([ct, total.to_frame().T])
    return ct


# ---------------------------------------------------------------------------
# 4. СОЗДАНИЕ EXCEL
# ---------------------------------------------------------------------------
def create_excel(all_data: dict, filename: str = "lab5_invoices.xlsx"):
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        for v, df in all_data.items():
            sheet_name = f"Вариант {v}"
            df.to_excel(writer, sheet_name=sheet_name, index=False)


# ---------------------------------------------------------------------------
# 5. СОЗДАНИЕ WORD
# ---------------------------------------------------------------------------
def add_table(doc, headers, rows):
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = "Table Grid"
    hdr = table.rows[0].cells
    for i, h in enumerate(headers):
        hdr[i].text = str(h)
    for row in rows:
        cells = table.add_row().cells
        for i, val in enumerate(row):
            cells[i].text = str(val)
    return table


def create_word(all_data: dict, filename: str = "lab5_answers.docx"):
    doc = Document()
    doc.add_heading("Лабораторная работа №5. Выборочное наблюдение", level=0)

    for v in range(1, 11):
        df = all_data[v]
        gen = compute_stats(df)
        mech = mechanical_sample(df, 50)
        rand = random_sample(df, 50, seed=100 + v)
        s_mech = sample_stats(mech)
        s_rand = sample_stats(rand)
        # для заданий 2–5 используем случайную выборку
        s = s_rand
        req_n = required_sample_size(gen["var"], 1000, t=2.0)
        risk = risk_level(rand)
        pivot = pivot_status_violations(rand)

        doc.add_heading(f"Вариант {v}. {VARIANTS[v]['feature']}", level=1)

        # Сэмпл первых 5 счетов
        doc.add_heading("Сэмпл первых десяти счетов", level=2)
        sample5 = df.head(5)
        add_table(doc,
                  ["Номер счёта", "Дата", "Контрагент", "Сумма счёта",
                   "Статус оплаты", "Наличие нарушений"],
                  sample5.values.tolist())

        # Задание 1
        doc.add_heading("Задание 1. Формирование выборки", level=2)
        doc.add_paragraph("Ожидаемые результаты:")
        add_table(doc,
                  ["Показатель", "Генеральная"],
                  [
                      ["Средняя сумма", f"{gen['mean']:.0f}"],
                      ["Дисперсия", f"{gen['var']:.0f}"],
                      ["Стандартное отклонение", f"{gen['std']:.0f}"],
                      ["Минимум", f"{gen['min']}"],
                      ["Максимум", f"{gen['max']}"],
                      ["Vsigma", f"{gen['std']/gen['mean']*100}%"],
                  ])


        # Задание 3
        doc.add_heading("Задание 3. Определение необходимого объёма выборки", level=2)
        doc.add_paragraph(f"Необходимый объём выборки = {req_n}")
        doc.add_paragraph("Если уменьшить точность (Δ = 5 000 руб.):")
        req_n_5000 = required_sample_size(gen["var"], 5000, t=2.0)
        doc.add_paragraph(f"Необходимый объём выборки = {req_n_5000}")

        # Задание 4
        doc.add_heading("Задание 4. Использование функции ЕСЛИ", level=2)
        doc.add_paragraph("Итоговая таблица")
        add_table(doc,
                  ["Уровень риска", "Количество", "Доля, %"],
                  [
                      ["Высокий", risk["high"], risk["high_pct"]],
                      ["Средний", risk["medium"], risk["medium_pct"]],
                      ["Низкий", risk["low"], risk["low_pct"]],
                      ["Итого", risk["high"] + risk["medium"] + risk["low"], 100.0],
                  ])
        doc.add_paragraph(
            "Вывод: примерно 20% счетов в выборке относятся к высокому риску. "
            "При распространении на 500 счетов это даёт ≈ 100 счетов высокого риска — "
            "они требуют немедленной проверки."
        )

        # Задание 5
        doc.add_heading("Задание 5. Сводная таблица", level=2)
        doc.add_paragraph("Ожидаемые результаты:")
        add_table(doc,
                  ["Статус оплаты", "Нарушения: Да", "Нарушения: Нет", "Всего"],
                  [[idx] + list(row) for idx, row in zip(pivot.index, pivot.values)])

        doc.add_page_break()

    doc.save(filename)


# ---------------------------------------------------------------------------
# 6. MAIN
# ---------------------------------------------------------------------------
def main():
    all_data = {}
    for v in range(1, 11):
        df = generate_invoices(v, seed=1000 + v)
        all_data[v] = df
        print(f"Вариант {v}: сгенерировано {len(df)} счетов, "
              f"средняя = {df['Сумма счёта'].mean():.0f}, "
              f"доля нарушений = {(df['Наличие нарушений']=='Да').mean():.2%}, "
              f"доля неоплаченных = {(df['Статус оплаты']=='Не оплачен').mean():.2%}")

    create_excel(all_data, "lab5_invoices.xlsx")
    print("Создан файл lab5_invoices.xlsx")

    create_word(all_data, "lab5_answers.docx")
    print("Создан файл lab5_answers.docx")


if __name__ == "__main__":
    main()

величина дисперсии при генерации данных варианта 1 = 223000000.0
Вариант 1: сгенерировано 500 счетов, средняя = 56422, доля нарушений = 20.00%, доля неоплаченных = 15.00%
величина дисперсии при генерации данных варианта 2 = 82000000.0
Вариант 2: сгенерировано 500 счетов, средняя = 48979, доля нарушений = 10.00%, доля неоплаченных = 10.00%
величина дисперсии при генерации данных варианта 3 = 534000000.0
Вариант 3: сгенерировано 500 счетов, средняя = 57953, доля нарушений = 30.00%, доля неоплаченных = 25.00%
величина дисперсии при генерации данных варианта 4 = 300000000.0
Вариант 4: сгенерировано 500 счетов, средняя = 60208, доля нарушений = 40.00%, доля неоплаченных = 30.00%
величина дисперсии при генерации данных варианта 5 = 142000000.0
Вариант 5: сгенерировано 500 счетов, средняя = 54236, доля нарушений = 5.00%, доля неоплаченных = 5.00%
величина дисперсии при генерации данных варианта 6 = 1710000000.0
Вариант 6: сгенерировано 500 счетов, средняя = 78617, доля нарушений = 20.00%, дол